In [ ]:
!cp -r /content/drive/MyDrive/Stage2 /content/Stage2

In [ ]:
import torch
from GRUdes import GAPGRU
from ultralytics import YOLO

model = YOLO("best.pt")  # load a pretrained model (recommended for training)

#Hook layers 10, 16, 19, 22

In [ ]:
selected_layers = [10]   # example indices
feature_dict = {}

def make_hook(name):
    def hook(module, input, output):
        #feature_dict[name] = output.detach().cpu()
        pooled = output.mean(dim=[2,3]).squeeze(0).cpu()
        feature_dict[name] = pooled.detach().numpy()
    return hook

for idx in selected_layers:
    model.model.model[idx].register_forward_hook(make_hook(f"layer_{idx}"))
    print(f"Hook registered for layer {idx}")

Labeling

In [ ]:
model.model.eval()

In [ ]:
import os
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader

Fight_train_dir = "/content/Stage2/RWF2000-2s-3fps/train/Fight"
NonFight_train_dir = "/content/Stage2/RWF2000-2s-3fps/train/NonFight"
Fight_val_dir = "/content/Stage2/RWF2000-2s-3fps/val/Fight"
NonFight_val_dir = "/content/Stage2/RWF2000-2s-3fps/val/NonFight"

sequence_length = 8  # Number of frames per video sequence

class VideoDataset(Dataset):
    def __init__(self, fight_dir, nonfight_dir, transform=None):
        self.fight_videos = [os.path.join(fight_dir, f) for f in os.listdir(fight_dir)[:10] if f.endswith('.avi')]
        self.nonfight_videos = [os.path.join(nonfight_dir, f) for f in os.listdir(nonfight_dir)[:10] if f.endswith('.avi')]
        self.videos = self.fight_videos + self.nonfight_videos
        self.labels = [1] * len(self.fight_videos) + [0] * len(self.nonfight_videos)
        self.transform = transform

    def __len__(self):
        return len(self.videos)
    
    def __getitem__(self, idx):
        video_path = self.videos[idx]
        label = self.labels[idx]
        
        cap = cv2.VideoCapture(video_path)
        frames = []
        count = 0
        while count < sequence_length:
            ret, frame = cap.read()
            if not ret:
                break
            with torch.no_grad():
                model(frame)

            frame_feature = torch.cat(
                [feature_dict[f"layer_{i}"] for i in selected_layers],
                dim=1
            )

            frames.append(frame_feature.squeeze(0))
            feature_dict.clear()
            count += 1

        if len(frames) < sequence_length:
            feature_dim = frames[0].shape[0]
            pad_len = sequence_length - len(frames)
            frames += [torch.zeros(feature_dim) for _ in range(pad_len)]
            
        cap.release()
        
        if self.transform:
            frames = [self.transform(frame) for frame in frames]
        
        return torch.stack(frames), label  #[num_frames, feature_dim], label
    def save_features(self, features, labels, save_path):
        np.savez(save_path, features=features, labels=labels)

Save preprocess data